# Step 6 - Deployment package and serving contract

**CSC3119 Assignment 4.** The brief requires the pipeline to run on RunPod inside **$5**. This notebook produces the deployment package, proves the online and offline feature builders agree, and aggregates every metric into `outputs/metrics.json` - the file to screenshot as RunPod proof.

Two things this notebook gets right that are easy to get wrong:

1. **The serving code is self-contained.** RunPod has no `src/` folder, so `serving.py` re-implements the feature row instead of importing `aq_common`. A parity assertion proves the two are identical, so online and offline can never silently diverge.
2. **Missing weather is accepted.** Rain or not, temperature can be null in a request; the model's imputer handles it, so the API never 500s.

In [1]:
from pathlib import Path
import sys, json, shutil, textwrap, importlib
import numpy as np
import pandas as pd
import joblib

PROJECT = Path.cwd()
while not (PROJECT / "src" / "aq_common.py").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / "src"))
import aq_common as A

A.set_stage("deploy")
A.banner("STEP 6 / 6  -  Deployment package and serving contract")
LOG = A.PipelineLog("step6_deployment")
DEPLOY = A.DEPLOY_DIR
B = joblib.load(A.stage_dir("model") / "best_pm25_model.joblib")

# The serving layer must be self-contained (RunPod has no src/ folder), so we write
# it here rather than importing aq_common in production.
print(f"model   : {B['model_name']}  features={len(B['features'])}")
print(f"deploy  : {DEPLOY}")


  STEP 6 / 6  -  Deployment package and serving contract


2026-10-20 14:45:41 | INFO | === step6_deployment ===


model   : RandomForest  features=9
deploy  : C:\Users\hp\OneDrive\Desktop\3rd YEAR 1st SEM\AI Model Development\Assignment 4\deploy


## 1. Write the deployment files

In [2]:
FILES = {}

FILES["requirements.txt"] = '''fastapi>=0.111
uvicorn[standard]>=0.30
pydantic>=2
scikit-learn==1.8.0
pandas
numpy
joblib
'''

FILES["serving.py"] = '''# Self-contained prediction logic, shared by the API and the tests.
# It mirrors aq_common.build_feature_row so an online request and an offline
# forecast produce byte-identical feature vectors.
import numpy as np
import pandas as pd


def feature_row(device_name, dt, temp, humidity, latitude, longitude, network):
    t = pd.Timestamp(dt)
    h = float(t.hour)
    return {
        "temp": temp,
        "humidity": humidity,
        "latitude": latitude,
        "longitude": longitude,
        "month": float(t.month),
        "hour_sin": float(np.sin(2 * np.pi * h / 24)),
        "hour_cos": float(np.cos(2 * np.pi * h / 24)),
        "network_airqo": int(str(network).lower() == "airqo"),
        "device_name": str(device_name),
    }


def predict(bundle, device_name, dt, latitude, longitude,
            network="airqo", temp=None, humidity=None):
    row = feature_row(device_name, dt, temp, humidity, latitude, longitude, network)
    X = pd.DataFrame([row])[bundle["features"]]
    z = float(bundle["model"].predict(X)[0])
    pm = float(max(0.0, z * bundle["pm25_std"] + bundle["pm25_mean"]))
    code = int(np.digitize(pm, bundle["aqi_bins"][1:-1]))
    return {
        "pm25_z": round(z, 4),
        "pm25_ugm3": round(pm, 1),
        "aqi": bundle["aqi_labels"][code],
        "aqi_code": code,
    }
'''

FILES["app.py"] = '''import joblib
from fastapi import FastAPI
from pydantic import BaseModel
from serving import predict

BUNDLE = joblib.load("best_pm25_model.joblib")
app = FastAPI(title="AirQo PM2.5 predictor", version="1.0")


class Reading(BaseModel):
    device_name: str
    datetime: str
    latitude: float
    longitude: float
    network: str = "airqo"
    temp: float | None = None
    humidity: float | None = None


@app.get("/health")
def health():
    return {"status": "ok", "model": BUNDLE["model_name"], "features": BUNDLE["features"]}


@app.post("/predict")
def predict_endpoint(r: Reading):
    return predict(BUNDLE, r.device_name, r.datetime, r.latitude, r.longitude,
                   r.network, r.temp, r.humidity)
'''

FILES["Dockerfile"] = '''FROM python:3.13-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py serving.py devices.json best_pm25_model.joblib ./
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

FILES["runpod_start.sh"] = '''#!/usr/bin/env bash
set -e
pip install -q -r requirements.txt
nvidia-smi || true                 # screenshot this for the report
python -c "import sklearn; print('sklearn', sklearn.__version__)"
uvicorn app:app --host 0.0.0.0 --port 8000
'''

FILES["client_example.py"] = '''import requests

URL = "https://POD_ID-8000.proxy.runpod.net/predict"   # replace POD_ID
payload = {
    "device_name": "aq_66",
    "datetime": "2026-08-01 08:00",
    "latitude": 0.3223,
    "longitude": 32.5758,
    "network": "airqo",
    "temp": 22.0,
    "humidity": 78.0,
}
print(requests.post(URL, json=payload, timeout=30).json())
'''

for name, content in FILES.items():
    (DEPLOY / name).write_text(content, encoding="utf-8")
    print(f"  wrote deploy/{name}")

  wrote deploy/requirements.txt
  wrote deploy/serving.py
  wrote deploy/app.py
  wrote deploy/Dockerfile
  wrote deploy/runpod_start.sh
  wrote deploy/client_example.py


In [3]:
# Device registry: the API needs to know where a sensor is when the caller does not.
reg = (pd.read_parquet(A.stage_dir("prep") / "hourly_clean.parquet")
       .query("is_observed").groupby("device_name", observed=True)
       .agg(site_name=("site_name", "first"), network=("network", "first"),
            latitude=("latitude", "median"), longitude=("longitude", "median"))
       .reset_index())
devices = {r.device_name: {"site_name": r.site_name, "network": r.network,
                           "latitude": (None if pd.isna(r.latitude) else round(r.latitude, 5)),
                           "longitude": (None if pd.isna(r.longitude) else round(r.longitude, 5))}
           for r in reg.itertuples()}
A.save_json(devices, DEPLOY / "devices.json")
print(f"wrote deploy/devices.json with {len(devices)} sensors")
shutil.copy(A.stage_dir("model") / "best_pm25_model.joblib", DEPLOY / "best_pm25_model.joblib")
print("copied best_pm25_model.joblib into deploy/ for the Docker image")

wrote deploy/devices.json with 149 sensors
copied best_pm25_model.joblib into deploy/ for the Docker image


## 2. Self-test the serving contract
We call the exact function the API calls, on three requests including one with null weather, and assert feature parity with the notebook path.

In [4]:
print("\n" + "="*78)
print(" 1. SELF-TEST THE SERVING CONTRACT (without FastAPI)")
print("="*78)

sys.path.insert(0, str(DEPLOY))
import serving
importlib.reload(serving)

samples = [
    dict(device_name="aq_66", dt="2026-08-01 08:00", latitude=0.3223, longitude=32.5758,
         network="airqo", temp=22.0, humidity=78.0),
    dict(device_name="aq_g5_37", dt="2026-08-01 18:00", latitude=0.3180, longitude=32.5899,
         network="airqo", temp=26.5, humidity=60.0),
    dict(device_name="ag_588c81267534", dt="2026-08-03 07:00", latitude=0.3335,
         longitude=32.5686, network="airgradient", temp=None, humidity=None),  # nulls OK
]
served = []
for s in samples:
    out = serving.predict(B, **s)
    served.append({"request": s, "response": out})
    print(f"  {s['device_name']:18s} {s['dt']}  ->  {out['pm25_ugm3']:5.1f} ug/m3  "
          f"({out['pm25_z']:+.3f} z, {out['aqi']})")
print("\nNull weather is accepted: the model's imputer handles it, so the API never")
print("crashes on a missing temperature.")

# Parity check: does the serving feature row equal the training/forecast one?
probe = serving.feature_row("aq_66", "2026-08-01 08:00", 22.0, 78.0, 0.3223, 32.5758, "airqo")
ref = A.build_feature_row("aq_66", "2026-08-01 08:00", 22.0, 78.0, 0.3223, 32.5758, "airqo")
assert probe == ref, "serving feature row drifted from aq_common.build_feature_row"
print("Parity check passed: the API builds the same feature vector as the notebook.")
A.save_json(served, A.stage_dir("deploy") / "serving_selftest.json")


 1. SELF-TEST THE SERVING CONTRACT (without FastAPI)
  aq_66              2026-08-01 08:00  ->   31.4 ug/m3  (-0.041 z, Moderate)
  aq_g5_37           2026-08-01 18:00  ->   19.7 ug/m3  (-0.460 z, Moderate)
  ag_588c81267534    2026-08-03 07:00  ->   81.1 ug/m3  (+1.742 z, Unhealthy)

Null weather is accepted: the model's imputer handles it, so the API never
crashes on a missing temperature.
Parity check passed: the API builds the same feature vector as the notebook.


## 3. Baseten / Truss package (always-on alternative)

In [5]:
print("\n" + "="*78)
print(" 2. BASETEN / TRUSS PACKAGE (alternative always-on host)")
print("="*78)

TRUSS = DEPLOY / "pm25-airqo-truss"
(TRUSS / "model").mkdir(parents=True, exist_ok=True)
(TRUSS / "config.yaml").write_text(
    "model_name: pm25-airqo\n"
    "python_version: py311\n"
    "requirements:\n"
    "  - scikit-learn==1.8.0\n"
    "  - pandas\n"
    "  - numpy\n"
    "  - joblib\n"
    "resources:\n"
    "  cpu: '2'\n"
    "  memory: 4Gi\n", encoding="utf-8")
(TRUSS / "model" / "model.py").write_text(
    "import joblib\n"
    "import numpy as np\n"
    "import pandas as pd\n\n"
    "BUNDLE = joblib.load('/app/model/best_pm25_model.joblib')\n\n\n"
    "def _row(d):\n"
    "    dt = pd.Timestamp(d['datetime'])\n"
    "    h = float(dt.hour)\n"
    "    return {'temp': d.get('temp'), 'humidity': d.get('humidity'),\n"
    "            'latitude': d['latitude'], 'longitude': d['longitude'],\n"
    "            'month': float(dt.month), 'hour_sin': float(np.sin(2*np.pi*h/24)),\n"
    "            'hour_cos': float(np.cos(2*np.pi*h/24)),\n"
    "            'network_airqo': int(str(d.get('network', 'airqo')).lower() == 'airqo'),\n"
    "            'device_name': str(d['device_name'])}\n\n\n"
    "class Model:\n"
    "    def predict(self, model_input):\n"
    "        rows = model_input if isinstance(model_input, list) else [model_input]\n"
    "        X = pd.DataFrame([_row(r) for r in rows])[BUNDLE['features']]\n"
    "        z = BUNDLE['model'].predict(X)\n"
    "        pm = np.clip(z * BUNDLE['pm25_std'] + BUNDLE['pm25_mean'], 0, None)\n"
    "        out = []\n"
    "        for zi, pi in zip(z, pm):\n"
    "            code = int(np.digitize(pi, BUNDLE['aqi_bins'][1:-1]))\n"
    "            out.append({'pm25_z': float(zi), 'pm25_ugm3': round(float(pi), 1),\n"
    "                        'aqi': BUNDLE['aqi_labels'][code]})\n"
    "        return out\n", encoding="utf-8")
shutil.copy(A.stage_dir("model") / "best_pm25_model.joblib", TRUSS / "model" / "best_pm25_model.joblib")
for f in sorted(TRUSS.rglob("*")):
    if f.is_file():
        print("  ", f.relative_to(DEPLOY))


 2. BASETEN / TRUSS PACKAGE (alternative always-on host)


   pm25-airqo-truss\config.yaml
   pm25-airqo-truss\model\best_pm25_model.joblib
   pm25-airqo-truss\model\model.py


## 4. Operations README and RunPod budget

In [6]:
README = '''# AirQo PM2.5 predictor - deployment package

Trained model: Random Forest pipeline (refit on all labelled sensor-hours).
Serving is CPU-only; a GPU pod is only used because the assignment runs on RunPod.

## Files
| File | Purpose |
|---|---|
| serving.py | self-contained feature build + predict (no aq_common needed) |
| app.py | FastAPI app: GET /health, POST /predict |
| devices.json | sensor registry (name -> site, network, lat, lon) |
| best_pm25_model.joblib | the model bundle |
| Dockerfile | container image for any host |
| runpod_start.sh | one-shot bootstrap + launch |
| client_example.py | example request |
| pm25-airqo-truss/ | Baseten/Truss package for always-on hosting |

## RunPod (budget well under $5)
1. Deploy a Community Cloud pod with the PyTorch template, cheapest available GPU
   (~$0.27-$0.50/h), container disk 20 GB, and expose HTTP port 8000.
2. Upload this folder; then `bash runpod_start.sh`.
3. Take the screenshots the report needs: pod summary, `nvidia-smi`, the notebook
   output, `tail outputs/runs/pipeline_log.txt`, and `outputs/metrics.json`.
4. Test: `python client_example.py` with POD_ID replaced, or open /docs.
5. **Terminate** the pod (stopping still bills for disk).

## Run the full pipeline on the pod (as the brief asks)
`jupyter nbconvert --to notebook --execute Step0_Run_All.ipynb --ExecutePreprocessor.timeout=3600`

## Cost estimate
| Item | Estimate |
|---|---|
| GPU pod, ~1 h (setup + run + downloads) | $0.27 - $0.50 |
| 20 GB container disk for a day | ~$0.07 |
| Safety margin for a second attempt | ~$0.50 |
| **Total** | **~$1, under the $5 limit** |
'''
(DEPLOY / "README.md").write_text(README, encoding="utf-8")
print("wrote deploy/README.md")
print("\nDeploy package contents:")
for f in sorted(DEPLOY.iterdir()):
    print("  ", f.name, "(dir)" if f.is_dir() else f"({f.stat().st_size:,d} bytes)")

wrote deploy/README.md

Deploy package contents:
   __pycache__ (dir)
   app.py (750 bytes)
   best_pm25_model.joblib (47,350,870 bytes)
   client_example.py (353 bytes)
   devices.json (21,769 bytes)
   Dockerfile (257 bytes)
   pm25-airqo-truss (dir)
   README.md (1,658 bytes)
   requirements.txt (98 bytes)
   runpod_start.sh (245 bytes)
   serving.py (1,357 bytes)


## 5. Aggregate `outputs/metrics.json` from every stage

In [7]:
print("\n" + "="*78)
print(" 3. AGGREGATE metrics.json FOR THE REPORT AND RUNPOD PROOF")
print("="*78)

def _json(p):
    p = Path(p)
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}

prep = _json(A.stage_dir("prep") / "prep_meta.json")
eda = _json(A.stage_dir("eda") / "eda_meta.json")
model = _json(A.stage_dir("model") / "model_meta.json")
fc = _json(A.stage_dir("forecast") / "forecast_meta.json")
mon = _json(A.stage_dir("monitor") / "monitoring_meta.json")

metrics = {
    "assignment": "CSC3119 Assignment 4 - PM2.5 prediction from AirQo data",
    "data": {"raw_rows": prep.get("raw_shape", [None])[0] if prep.get("raw_shape") else None,
             "sensors": prep.get("n_sensors"), "sites": prep.get("n_sites"),
             "grid_rows": prep.get("grid_rows"),
             "modelling_rows": prep.get("modelling_rows"),
             "modelling_sensors": prep.get("modelling_sensors"),
             "sensors_no_readings": prep.get("sensors_no_readings"),
             "null_readings_dropped": prep.get("null_readings_dropped"),
             "observed_share": (prep.get("observed_share_pct") or 0) / 100.0,
             "observed_share_pct": prep.get("observed_share_pct"),
             "windows": prep.get("dates_present"),
             "pm25_mean": eda.get("zscore", {}).get("mean"),
             "pm25_std": eda.get("zscore", {}).get("std")},
    "model": {"selected": model.get("selected"),
              "selection_rule": model.get("selection_rule"),
              "train_rows": model.get("train_rows"), "test_rows": model.get("test_rows"),
              "unhealthy_threshold_ugm3": model.get("unhealthy_threshold_ugm3"),
              "error_margins_95ci": model.get("error_margins_95ci"),
              "pm10_leakage_ablation": model.get("pm10_leakage_ablation"),
              "holdout": B["test_metrics"]},
    "model_comparison": model.get("comparison"),
    "feature_importance": model.get("permitted_importance") or model.get("feature_importance"),
    "forecast": {"n_sensors": fc.get("n_sensors"), "n_hours": fc.get("n_hours"),
                 "bands": fc.get("sensor_day_band_counts")},
    "monitoring": {"runA_drift_share": mon.get("runA", {}).get("drift_share"),
                   "runB_drift_share": mon.get("runB", {}).get("drift_share"),
                   "reports": mon.get("reports")},
    "env": A.env_facts(),
}
A.save_json(metrics, A.OUT_DIR / "metrics.json")
A.save_json(metrics, PROJECT / "outputs" / "metrics.json")

print("Headline metrics:")
h = B["test_metrics"]
print(f"  best model        : {model.get('selected')}")
print(f"  hold-out RMSE     : {h.get('RMSE (ug/m3)'):.2f} ug/m3  (z {h.get('RMSE (z)'):.3f})")
print(f"  hold-out MAE      : {h.get('MAE (ug/m3)'):.2f} ug/m3")
print(f"  hold-out R2       : {h.get('R2'):.3f}")
print(f"  hold-out ROC-AUC  : {h.get('ROC-AUC', float('nan')):.3f}")
print(f"  AQI band accuracy : {h.get('AQI accuracy')*100:.1f}%")
print(f"  forecast sensors  : {fc.get('n_sensors')} x {fc.get('n_hours')} h")
print("wrote outputs/metrics.json")
LOG.save({"stage": "deployment", "selected": model.get("selected"),
          "files": len(list(DEPLOY.iterdir()))})


 3. AGGREGATE metrics.json FOR THE REPORT AND RUNPOD PROOF
Headline metrics:
  best model        : RandomForest
  hold-out RMSE     : 22.04 ug/m3  (z 0.791)
  hold-out MAE      : 14.50 ug/m3
  hold-out R2       : 0.234
  hold-out ROC-AUC  : 0.812
  AQI band accuracy : 61.4%
  forecast sensors  : 149 x 384 h
wrote outputs/metrics.json


{'run': 'step6_deployment',
 'generated': '2026-10-20T14:45:44',
 'python': '3.13.5',
 'pandas': '3.0.2',
 'host': 'SHANE',
 'cpu_count': 4,
 'stage_times_s': {},
 'total_runtime_s': 5.5,
 'stage': 'deployment',
 'selected': 'RandomForest',
 'files': 11}

In [8]:
print("\n" + "="*78)
print("  STEP 6 COMPLETE - DELIVERABLES")
print("="*78)
manifest = {
    "notebooks": [p.name for p in sorted(A.NOTEBOOKS_DIR.glob("*.ipynb"))],
    "outputs": {d.name: [f.name for f in sorted(d.glob("*"))]
                for d in sorted(A.OUT_DIR.iterdir()) if d.is_dir()},
    "model_file": (A.stage_dir("model") / "best_pm25_model.joblib").exists(),
    "deploy_files": [f.name for f in sorted(DEPLOY.iterdir())],
}
for k, v in manifest["outputs"].items():
    print(f"  outputs/{k}: {len(v)} files")
print(f"  notebooks    : {len(manifest['notebooks'])}")
print(f"  deploy files : {len(manifest['deploy_files'])}")
A.save_json(manifest, A.stage_dir("deploy") / "deliverables_manifest.json")
print("\nRun order for the whole assignment:")
print("  Step1 -> Step2 -> Step3 -> Step4 -> Step5 -> Step6  (or Step0_Run_All.ipynb)")


  STEP 6 COMPLETE - DELIVERABLES
  outputs/01_prep: 16 files
  outputs/02_eda: 14 files
  outputs/03_model: 14 files
  outputs/04_forecast: 9 files
  outputs/05_monitor: 8 files
  outputs/06_deploy: 2 files
  outputs/runs: 8 files
  notebooks    : 7
  deploy files : 11

Run order for the whole assignment:
  Step1 -> Step2 -> Step3 -> Step4 -> Step5 -> Step6  (or Step0_Run_All.ipynb)
